# Debug Notebook: Simulate Player Using main.py

This notebook allows you to interact with the FastAPI card game server in `main.py` as if you were a player. It demonstrates connecting to the server, submitting a deck, and using the websocket for real-time communication.

In [33]:
# Import Required Libraries
import sys
import importlib
import requests
import asyncio
import websockets
import json
from pathlib import Path

In [ ]:
# 3. Persistent WebSocket interaction (async)
class PersistentWebSocket:
    def __init__(self, uri):
        self.uri = uri
        self.websocket = None

    async def connect(self):
        self.websocket = await websockets.connect(self.uri)
        greeting = await self.websocket.recv()
        print("WebSocket server says:", greeting)

    async def send(self, txt):
        await self.websocket.send(txt)
        response = await self.websocket.recv()
        print("WebSocket echo:", response)

    async def close(self):
        await self.websocket.close()

# Usage example:
# ws = PersistentWebSocket(f"ws://localhost:8000/ws/{game_id}/{player_id}")
# await ws.connect()
# await ws.send("card dwa223_azrzad played")
# await ws.close()

In [34]:
# Load main.py as a Module
main_path = Path("main.py").resolve()
if str(main_path.parent) not in sys.path:
    sys.path.insert(0, str(main_path.parent))
main_module = importlib.import_module("main")
# Now you can access main_module.main or other functions/classes if needed

In [ ]:
# Simulate Player Actions Using main.py Functions
# Set up test player and game IDs
game_id = "testgame1"
player_id = "player1"
base_url = "http://127.0.0.1:8000"

# 1. Connect player
connect_resp = requests.post(f"{base_url}/connect", json={"player_id": player_id, "game_id": game_id})
print("Connect response:", connect_resp.json())

# 2. Submit deck (using example card IDs)
deck = ["fireball", "fireball", "fireball", "fireball", "fireball", "fireball"]
deck_resp = requests.post(f"{base_url}/submit_deck", json={"player_id": player_id, "game_id": game_id, "deck": deck})
print("Deck submission response:", deck_resp.json())

Connect response: {'status': 'connected'}
Deck submission response: {'status': 'deck submitted'}


In [45]:
# Display Output and Debug Information
# Run the websocket interaction
ws = PersistentWebSocket(f"ws://127.0.0.1:8000/ws/{game_id}/{player_id}")
await ws.connect()

WebSocket server says: Connected to game.


In [46]:
await ws.send("card dwa223_azrzad played")
await ws.send("another message")
await ws.close()

WebSocket echo: Echo: card dwa223_azrzad played
WebSocket echo: Echo: another message


In [ ]:
# Test Edge Cases and Error Handling
# Try connecting with an invalid game_id or missing deck
invalid_resp = requests.post(f"{base_url}/connect", json={"player_id": "", "game_id": ""})
print("Invalid connect response:", invalid_resp.status_code, invalid_resp.text)

invalid_deck_resp = requests.post(f"{base_url}/submit_deck", json={"player_id": player_id, "game_id": game_id, "deck": []})
print("Invalid deck submission response:", invalid_deck_resp.status_code, invalid_deck_resp.text)